In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import os

In [24]:
data = pd.read_csv('../DMS_substitutions.csv')
merged = data.groupby('UniProt_ID', as_index=False).agg(lambda x: '; '.join(map(str, x.unique())))
#merged = data.groupby('UniProt_ID', as_index=False).agg(lambda x: '; '.join(map(str, x)))
merged['num_selections'] = merged['DMS_filename'].str.split(';').str.len()
merged['num_groups'] = merged['first_author'].str.split(';').apply(lambda x: len(set(x)))
merged['expression'] = merged['coarse_selection_type'].apply(lambda x: len(x.split('Expression'))-1)
merged['activity'] = merged['coarse_selection_type'].apply(lambda x: len(x.split('Activity'))-1)
merged['stability'] = merged['coarse_selection_type'].apply(lambda x: len(x.split('Stability'))-1)
merged['fitness'] = merged['coarse_selection_type'].apply(lambda x: len(x.split('OrganismalFitness'))-1)
merged['binding'] = merged['coarse_selection_type'].apply(lambda x: len(x.split('Binding'))-1)

### Groups:
* Group1 (ProteinGym): Same publication, multiple selections
    * subgrouped by publication
* Group2 (ProteinGym): Multiple groups, same selection
* Group3 (ProteinGym): Same group, multiple selections, some selections are de novo
* Group4 (other): Wildcard; De-novo optimizations

In [ ]:
def make_group_dfs(data):
    rows = []
    for up, dfp in data.groupby('UniProt_ID'):
        out = {
            'UniProt_ID': up,
            'group1_member': 0,
            'group1_str': '',
            'group2_member': 0,
            'group2_str': ''
        }
        g1_strs = [] # ---------- Group 1 ----------
        g1_cols = ['first_author', 'title', 'year', 'jo']
        for _, g in dfp.groupby(g1_cols):
            if g['coarse_selection_type'].nunique() > 1:
                out['group1_member'] = 1
                s = (g['DMS_filename'] + ':' + g['coarse_selection_type']) \
                        .unique()
                g1_strs.append(','.join(s))
        out['group1_str'] = ';'.join(g1_strs)
        g2_strs = [] # ---------- Group 2 ----------
        for _, g in dfp.groupby('coarse_selection_type'):
            if g[g1_cols].drop_duplicates().shape[0] > 1:
                out['group2_member'] = 1
                s = (g['DMS_filename'] + ':' + g['coarse_selection_type']) \
                        .unique()
                g2_strs.append(','.join(s))
        out['group2_str'] = ';'.join(g2_strs)
        rows.append(out)
    return pd.DataFrame(rows)

group_df = make_group_dfs(data)
group_df = group_df[(group_df['group1_member']!=0) | (group_df['group2_member']!=0)]
group_df.to_csv('DMS_substitutions_groups.tsv',sep='\t',index=False)

### Making group 3 CSVs

For AIME_PSEAE...
* make mutation column
* merge dataframes
* save to csvs

In [ ]:
filename = '../raw_new_mutation_data/AMIE_PSEAE_amiESelectionFitnessData_Acetamide.txt'
df = pd.read_csv(filename,sep='\t')
df['normalized_fitness'] = pd.to_numeric(df['normalized_fitness'], errors='coerce')
df = df.dropna()
seq = 'MRHGDISSSNDTVGVAVVNYKMPRLHTAAEVLDNARKIAEMIVGMKQGLPGMDLVVFPEYSLQGIMYDPAEMMETAVAIPGEETEIFSRACRKANVWGVFSLTGERHEEHPRKAPYNTLVLIDNNGEIVQKYRKIIPWCPIEGWYPGGQTYVSEGPKGMKISLIICDDGNYPEIWRDCAMKGAELIVRCQGYMYPAKDQQVMMAKAMAWANNCYVAVANAAGFDGVYSYFGHSAIIGFDGRTLGECGEEEMGIQYAQLSLSQIRDARANDQSQNHLFKILHRGYSGLQASGDGDRGLAECPFEFYRTWVTDAEKARENVERLTRSTTGVAQCPVGRLPYEGLEKEA'

def center_wt(group):
    wt = group.loc[group['mutation'] == '*', 'normalized_fitness']
    if wt.empty:
        return group  # or raise / drop
    wt_val = wt.iloc[0]
    group['normalized_fitness_centered'] = group['normalized_fitness'] / wt_val
    return group

df = df.groupby('location', group_keys=False).apply(center_wt)


/tmp/ipykernel_2626906/205696676.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('location', group_keys=False).apply(center_wt)


,location,mutation,normalized_fitness,normalized_fitness_rep1,normalized_fitness_rep2,unsel_reads,sel_reads,sel_reads_rep1,sel_reads_rep2,rawlog2,rawlog2_rep1,rawlog2_rep2,normalized_fitness_centered
0,1,*,-2.7170,-3.0724,-2.5573,99.0,3.0,1.0,2.0,-6.631586,-6.961172,-6.433445,1.000000
1,1,F,-1.7829,-1.6369,-1.9291,28.0,2.0,1.0,1.0,-5.394547,-5.139171,-5.611444,0.656202
4,1,P,-2.1514,-1.9631,-2.3465,84.0,4.0,2.0,2.0,-5.979509,-5.724133,-6.196406,0.791829
6,1,I,-0.1227,-0.0893,-0.1518,89.0,310.0,150.0,160.0,0.213199,0.421270,0.042106,0.045160
7,1,L,-0.3360,-0.3522,-0.3256,74.0,118.0,47.0,71.0,-0.914002,-0.986680,-0.863795,0.123666
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7156,341,D,0.1300,0.1057,0.1423,179.0,1742.0,556.0,1186.0,1.711717,1.545290,1.796881,0.256360
7157,341,E,0.0272,-0.0237,0.0508,33.0,202.0,58.0,144.0,1.042821,0.723752,1.194340,0.053638
7158,341,H,-0.1865,-0.2591,-0.1545,78.0,201.0,55.0,146.0,-0.205347,-0.593878,-0.026769,-0.367778
7159,341,K,0.0378,0.0471,0.0325,39.0,250.0,93.0,157.0,1.109386,1.163921,1.078027,0.074542


In [14]:
pg_df = pd.read_csv('../DMS_ProteinGym_substitutions/AMIE_PSEAE_Wrenbeck_2017.csv')
pg_df['mutated_sequence'].iloc[0]

'HRHGDISSSNDTVGVAVVNYKMPRLHTAAEVLDNARKIAEMIVGMKQGLPGMDLVVFPEYSLQGIMYDPAEMMETAVAIPGEETEIFSRACRKANVWGVFSLTGERHEEHPRKAPYNTLVLIDNNGEIVQKYRKIIPWCPIEGWYPGGQTYVSEGPKGMKISLIICDDGNYPEIWRDCAMKGAELIVRCQGYMYPAKDQQVMMAKAMAWANNCYVAVANAAGFDGVYSYFGHSAIIGFDGRTLGECGEEEMGIQYAQLSLSQIRDARANDQSQNHLFKILHRGYSGLQASGDGDRGLAECPFEFYRTWVTDAEKARENVERLTRSTTGVAQCPVGRLPYEGLEKEA'